Imports

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
from pathlib import Path
import smtplib
import os
from email.mime.text import MIMEText

Pull classes (do this once)

In [2]:
url = "https://www.oldtownschool.org/classes/adults/ensemble/"

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; ClassesListed/1.0)"
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

Class data to data frame

In [33]:
rows = []

# each class block
class_blocks = soup.find_all("div", class_="content")

for block in class_blocks:

    # ---------- TITLE ----------
    h4 = block.find("h4")

    if h4:
        full_title = h4.get_text(" ", strip=True)

        if " with " in full_title:
            class_name = full_title.split(" with ")[0]
            teacher = full_title.split(" with ")[1]
        else:
            class_name = full_title
            teacher = None
    else:
        class_name = None
        teacher = None

    # ---------- PRICE ----------
    price_div = block.find("div", class_="price")

    price = (
        price_div.get_text(" ", strip=True)
        if price_div else None
    )

    # ---------- DATES ----------
    dates_div = block.find("div", class_="dates")

    dates = (
        dates_div.get_text(" ", strip=True)
        if dates_div else None
    )

    # ---------- TIMES ----------
    times_div = block.find("div", class_="times")

    times = (
        times_div.get_text(" ", strip=True)
        if times_div else None
    )

    # ---------- DESCRIPTION ----------
    panel_body = block.find("div", class_="panel-body")

    description = (
        panel_body.get_text(" ", strip=True)
        if panel_body else None
    )

    # ---------- INSTRUMENTS ----------
    panel_footer = block.find("div", class_="panel-footer")

    instruments = None

    if panel_footer:
        footer_text = panel_footer.get_text(" ", strip=True)

        if "Instruments:" in footer_text:
            instruments = footer_text.replace(
                "Instruments:",
                ""
            ).strip()

    # ---------- REGISTER LINK ----------
    register_btn = block.find("a", class_="courselink")

    register_link = (
        register_btn["href"]
        if register_btn else None
    )

    # ---------- SAVE ROW ----------
    rows.append({
        "class_name": class_name,
        "teacher": teacher,
        "price": price,
        "dates": dates,
        "times": times,
        "description": description,
        "instruments": instruments,
        "register_link": register_link
    })

# ---------- DATAFRAME ----------
df = pd.DataFrame(rows).dropna().drop_duplicates()

df.head()

,class_name,teacher,price,dates,times,description,instruments,register_link
1,‘60s Classics Ensemble,Steve Levitt,$231 ($214 members),06/27/2026 – 08/15/2026 (7 meetings),Saturdays · 11:00 AM – 12:20 PM · In-Person @4...,You don’t have to be 60 to be in the ‘60s Clas...,"Bass, Drum Kit, Fiddle, Guitar, Piano, Voice",/cart/classes/AddToCart.php?class=171762
4,‘70s and ‘80s Feel Good Sensations Ensemble,Julia Storke · Scott Besaw,$280 ($260 members),06/24/2026 – 08/12/2026 (8 meetings),Wednesdays · 6:30 PM – 7:50 PM · In-Person @45...,"From Soul Train to American Bandstand , the so...","Bass, Clarinet, Drum Kit, Flute, Guitar, Saxop...",/cart/classes/AddToCart.php?class=171763
6,Alt-Country Ensemble,Shelley Miller,$231 ($214 members),06/27/2026 – 08/15/2026 (7 meetings),Saturdays · 1:00 PM – 2:20 PM · In-Person @454...,Grab your twang and get ready to rock. We'll l...,"Bass, Banjo, Dobro, Drum Kit, Fiddle, Flute, G...",/cart/classes/AddToCart.php?class=171764
8,New Americana Ensemble,Alton Smith,$264 ($244 members),06/22/2026 – 08/10/2026 (8 meetings),Mondays · 6:30 PM – 7:50 PM · In-Person @909 W...,Join us as we explore the songs of the ever-bu...,"Bass, Drum Kit, Fiddle, Guitar, Mandolin, Ukul...",/cart/classes/AddToCart.php?class=171785
10,Daytime Beatles Ensemble,Steve Levitt,$264 ($244 members),06/26/2026 – 08/14/2026 (8 meetings),Fridays · 12:00 PM – 1:20 PM · In-Person @4544...,A special daytime version of our most popular ...,"Bass, Drum Kit, Guitar, Piano, Voice",/cart/classes/AddToCart.php?class=171772


In [35]:
def get_classes():
    classes = df[["class_name", "dates"]].drop_duplicates()
    return classes

In [41]:
SEEN_FILE = "seen_classes.json"

def load_seen():

    if not Path(SEEN_FILE).exists():
        return set()

    try:
        with open(SEEN_FILE, "r") as f:
            data = json.load(f)

        return set(tuple(x) for x in data)

    except (json.JSONDecodeError, ValueError):

        # corrupted or empty file → reset
        return set()

def save_seen(seen):

    with open(SEEN_FILE, "w") as f:
        json.dump([list(x) for x in seen], f)

In [3]:
def send_email(new_classes):

    if not new_classes:
        return

    body = "New classes found:\n\n"

    for name, dates in new_classes:
        body += f"{name}\n{dates}\n\n"

    msg = MIMEText(body)
    msg["Subject"] = "New Class Alert"
    msg["From"] = os.environ["EMAIL_USER"]
    msg["To"] = os.environ["EMAIL_USER"]

    server = smtplib.SMTP("smtp.gmail.com", 587)
    server.starttls()

    server.login(
        os.environ["EMAIL_USER"],
        os.environ["EMAIL_PASS"]
    )

    server.send_message(msg)
    server.quit()

In [ ]:
classes = get_classes()

seen = load_seen()

new_classes = []

for _, row in classes.iterrows():

    key = (row["class_name"], row["dates"])

    if key not in seen:
        new_classes.append(key)

if new_classes:

    print("NEW CLASS FOUND:")

    for class_name, dates in new_classes:
        print("-", class_name)
        print(" ", dates)

    send_email(new_classes)

    seen.update(new_classes)
    save_seen(seen)

else:
    print("No new classes.")

NEW CLASS FOUND:
- ‘60s Classics Ensemble
  06/27/2026 – 08/15/2026  (7 meetings)
- ‘70s and ‘80s Feel Good Sensations Ensemble
  06/24/2026 – 08/12/2026  (8 meetings)
- Alt-Country Ensemble
  06/27/2026 – 08/15/2026  (7 meetings)
- New Americana Ensemble
  06/22/2026 – 08/10/2026  (8 meetings)
- Daytime Beatles Ensemble
  06/26/2026 – 08/14/2026  (8 meetings)
- The Beatles Ensemble
  06/24/2026 – 08/12/2026  (8 meetings)
- Blue Eyed Soul Ensemble
  06/22/2026 – 07/13/2026  (4 meetings)
- Bluegrass Ensemble
  06/22/2026 – 08/10/2026  (8 meetings)
- Blues Band Live!
  06/24/2026 – 08/12/2026  (8 meetings)
- Boygenius Ensemble 
  06/24/2026 – 08/12/2026  (8 meetings)
- Brazilian Ensemble
  06/23/2026 – 08/11/2026  (7 meetings)
- Classic Country Ensemble
  07/05/2026 – 08/16/2026  (7 meetings)
- Modern Country Ensemble
  06/23/2026 – 08/11/2026  (8 meetings)
- Deep Dive Ensemble
  06/24/2026 – 07/15/2026  (4 meetings)
- Folk Songs: Carrying History Forward
  06/25/2026 – 08/13/2026  (8 me